In [1]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv
import torch
from torch import nn
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(name)s | %(levelname)s | %(message)s",
)

torch.manual_seed(123)

src_path = Path.cwd().parent / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Added to sys.path: {src_path}")
load_dotenv()  # reads .env file from the current directory

PATH_DATA = Path.cwd().parent / ".data"
PATH_GPT2_124M_WEIGHTS = PATH_DATA / "model_weights" / "gpt2" / "124M"
PATH_GPT2_124M_WEIGHT_SETTINGS = PATH_GPT2_124M_WEIGHTS / "settings.pickle.gz"
PATH_GPT2_124M_WEIGHTS_PARAMETERS = PATH_GPT2_124M_WEIGHTS / "parameters.pickle.gz"
PATH_GPT2_CLASSIFIER_MODEL = PATH_DATA / "models" / "gpt2_classifier.pth"

DEFAULT_CONTEXT_LENGTH = 1024
DEFAULT_STRIDE = 1
DEFAULT_BATCH_SIZE=16


Added to sys.path: /home/jtv/code/jtviegas/languagemodels/src


In [2]:
import tiktoken
from tgedr_languagemodels.gpt2.classifier import GPT2Classifier
from tgedr_languagemodels.configuration import GPT2_MODEL_CONFIGS, BaseClassifierConfig, BaseModelConfig
from tgedr_languagemodels.gpt2.model import GPT2Model
from tgedr_languagemodels.utils.model_weights import load_weights_into_gpt

model_name = "gpt2-small (124M)"
tokenizer = tiktoken.get_encoding("gpt2")

config: BaseClassifierConfig = BaseClassifierConfig(
    vocabulary_size=tokenizer.n_vocab,
    context_length=DEFAULT_CONTEXT_LENGTH,
    embeddings_dimension=GPT2_MODEL_CONFIGS[model_name]["emb_dim"],
    n_heads=GPT2_MODEL_CONFIGS[model_name]["n_heads"],
    n_layers=GPT2_MODEL_CONFIGS[model_name]["n_layers"],
    drop_rate=0.0,
    qkv_bias=True,
    stride=DEFAULT_STRIDE,
    n_classes=3
)


classifier = GPT2Classifier(config)

In [3]:
from tgedr_languagemodels.utils.utils_llm import load_pickle_compressed

pretrained_weights = load_pickle_compressed(PATH_GPT2_124M_WEIGHTS_PARAMETERS)
classifier.pretrain(pretrained_weights)

In [4]:
from datasets import ClassLabel, Dataset, load_dataset

dataset = load_dataset("FinanceMTEB/financial_phrasebank", split="train")

httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/FinanceMTEB/financial_phrasebank/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
httpx | INFO | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/FinanceMTEB/financial_phrasebank/14efe6ac2635395e768682e0b91ce794b50c7ff3/README.md "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/FinanceMTEB/financial_phrasebank/resolve/14efe6ac2635395e768682e0b91ce794b50c7ff3/financial_phrasebank.py "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/FinanceMTEB/financial_phrasebank/FinanceMTEB/financial_phrasebank.py "HTTP/1.1 404 Not Found"
httpx | INFO | HTTP Request: GET https://huggingface.co/api/datasets/FinanceMTEB/financial_phrasebank/revision/14efe6ac2635395e768682e0b91ce794b50c7ff3 "HTTP/1.1 200 OK"
httpx | INFO | HTTP Request: HEAD https://huggingface.co/datasets/FinanceMTEB/financial_phraseba

In [5]:
from tgedr_languagemodels.utils.utils_data import ClassifierDataLoader

loader = ClassifierDataLoader(
    tokenizer=tokenizer,
    batch_size=DEFAULT_BATCH_SIZE,
).create(data=dataset)

In [6]:
classifier.fit(
    train_loader=loader["train"],
    val_loader=loader["validation"],
    num_epochs=12,
    eval_batches=12,
)

tgedr_languagemodels.gpt2.classifier | INFO | Training accuracy: 41.67% | Validation accuracy: 21.88%
tgedr_languagemodels.gpt2.classifier | INFO | Training accuracy: 61.98% | Validation accuracy: 44.27%


KeyboardInterrupt: 